# Lab 16: VEGAS tutorial

How accurately can we estimate an area from random samples?

We will start with a one-dimensional integral, then use the same ideas to integrate a two-dimensional function and generate events from it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf

plt.rcParams.update({'font.size': 12, 'axes.spines.top': False,
                     'axes.spines.right': False})


In [ ]:
def show_toy_samples(x, sigma):
    grid = np.linspace(0, 1, 5000)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.0), constrained_layout=True)
    for ax in axes:
        ax.fill_between(grid, toy(grid, sigma), color='#007c91', alpha=.15)
        ax.plot(grid, toy(grid, sigma), color='#007c91', label='g(x)')
        ax.scatter(x, toy(x, sigma), s=18, color='#d16a20', zorder=3, label='Sampled heights')
        ax.plot(x, np.zeros_like(x), '|', color='#526579', alpha=.65)
        ax.set(xlabel='x', ylabel='g(x)', ylim=(0, None))
    axes[0].set(xlim=(0, 1), title=f'{len(x)} uniform points; width = {sigma:g}')
    if sigma < .02:
        axes[1].set(xlim=(.63-5*sigma, .63+5*sigma), title='Zoom in on the peak')
    else:
        estimate, error = summarize_weights(toy(x, sigma))
        axes[1].axhline(estimate, color='#d16a20', ls='--', label='Sample mean')
        axes[1].set(xlim=(0, 1), title=f'Area estimate = {estimate:.3f} ± {error:.3f}')
    axes[0].legend(fontsize=10)
    plt.show()


def repeated_uniform(sigma, trials=40, size=100, seed=142):
    rng = np.random.default_rng(seed)
    return np.array([summarize_weights(toy(rng.random(size), sigma))
                     for _ in range(trials)])


def show_uniform_repeats(broad, narrow):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True, sharey=True)
    for ax, results, sigma in zip(axes, (broad, narrow), (.12, .003)):
        ax.errorbar(np.arange(len(results)), results[:, 0], yerr=results[:, 1],
                    fmt='o', ms=3, capsize=2, color='#007c91', lw=.8)
        ax.axhline(toy_integral(sigma), color='#d16a20', label='Exact area')
        ax.set(xlabel='Independent run', title=f'Width = {sigma:g}', ylabel='Estimated area')
    axes[0].legend(fontsize=10)
    plt.show()


def proposal_density(grid, edges):
    index = np.searchsorted(edges, grid, side='right')-1
    index = np.minimum(index, len(edges)-2)
    return 1/(len(np.diff(edges))*np.diff(edges)[index])


def show_bins(edges, x):
    grid = np.linspace(0, 1, 5000)
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), constrained_layout=True)
    axes[0].stairs(1/(len(edges)-1)/np.diff(edges), edges, color='#007c91', fill=True, alpha=.35)
    axes[0].plot(x, np.zeros_like(x), '|', color='#d16a20', alpha=.5)
    axes[0].set(xlabel='x', ylabel='q(x)', title='Equal probability in unequal intervals', xlim=(0, 1))
    axes[1].plot(grid, toy(grid, .003), color='#007c91', label='g(x)')
    axes[1].plot(grid, proposal_density(grid, edges), color='#d16a20', label='q(x)')
    axes[1].set(xlim=(.60, .66), xlabel='x', ylabel='Function / density', title='Peak region')
    axes[1].legend(fontsize=10)
    plt.show()


def show_weight_spread(uniform_weights, adaptive_weights):
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), constrained_layout=True, sharey=True)
    for ax, values, label in zip(axes, (uniform_weights, adaptive_weights), ('Uniform', 'Unequal bins')):
        ax.plot(np.arange(len(values)), values, '.', ms=3, alpha=.55, color='#007c91')
        ax.axhline(1, color='#d16a20', label='Exact area')
        ax.set(xlabel='Sample index', ylabel='Weight g(x) / q(x)', title=label)
    plt.show()


def show_grid_history(history):
    grid = np.linspace(0, 1, 5000)
    fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True, sharey=True)
    for ax, step in zip(axes, (0, 1, len(history)-1)):
        edges = history[step]
        ax.plot(grid, toy(grid, .003), color='#526579', lw=1.5, label='g(x)')
        ax.plot(grid, proposal_density(grid, edges), color='#007c91', label='q(x)')
        ax.plot(edges, np.zeros_like(edges), '|', color='#d16a20', markersize=9)
        ax.set(xlim=(.59, .67), xlabel='x', title=f'After {step} update' + ('' if step == 1 else 's'))
    axes[0].set_ylabel('Function / density')
    axes[0].legend(fontsize=10)
    plt.show()


def compare_equal_cost(trials=32, seed=142):
    uniform, adaptive = [], []
    for trial_seed in np.random.SeedSequence(seed).spawn(trials):
        train_seed, production_seed, uniform_seed = trial_seed.spawn(3)
        learned = train_1d(lambda x: toy(x, .003), seed=train_seed)
        a = estimate_1d(lambda x: toy(x, .003), learned['edges'], 2000, production_seed)
        u = estimate_1d(lambda x: toy(x, .003), np.array([0., 1.]),
                        learned['nTrainEval']+2000, uniform_seed)
        adaptive.append((a['estimate'], a['error']))
        uniform.append((u['estimate'], u['error']))
    return np.array(uniform), np.array(adaptive)


def show_equal_cost(uniform, adaptive):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True, sharey=True)
    truth = toy_integral(.003)
    for ax, results, title in zip(axes, (uniform, adaptive), ('8,000 uniform evaluations', '6,000 training + 2,000 production')):
        ax.errorbar(np.arange(len(results)), results[:, 0], yerr=results[:, 1],
                    fmt='o', ms=3, capsize=2, color='#007c91', lw=.8)
        ax.axhline(truth, color='#d16a20')
        rmse = np.sqrt(np.mean((results[:, 0]-truth)**2))
        ax.set(xlabel='Independent run', ylabel='Estimated area', title=f'{title}\nRMS error across runs = {rmse:.3f}')
    plt.show()


def show_integral_2d(uniform, adaptive, truth):
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), constrained_layout=True)
    results = (uniform, adaptive)
    axes[0].errorbar([0, 1], [r['estimate'] for r in results],
                     yerr=[r['error'] for r in results], fmt='o', capsize=6, color='#007c91')
    axes[0].axhline(truth, color='#d16a20', label='Exact finite-domain integral')
    axes[0].set(xticks=[0, 1], xticklabels=['Uniform', 'Learned grid'], ylabel='Integral estimate', xlim=(-.5, 1.5))
    axes[0].legend(fontsize=9)
    for r, label, color in zip(results, ('Uniform', 'Learned grid'), ('#526579', '#007c91')):
        axes[1].hist(r['weights'], bins=50, histtype='step', density=True, label=label, color=color)
    axes[1].set(xlabel='Weight f / q', ylabel='Density', title='Variation in production weights')
    axes[1].legend(fontsize=10)
    plt.show()


def show_damping():
    grid = np.linspace(.59, .67, 2000)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True, sharey=True)
    for ax, alpha, title in zip(axes, (1.5, None), ('Compression: alpha = 1.5', 'Compression removed')):
        ax.plot(grid, toy(grid, .003), color='#263d59', lw=2, label='g(x)')
        for seed in (140, 141, 142, 143):
            trained = train_1d(lambda x: toy(x, .003), batch=500, iterations=6,
                               alpha_damp=alpha, seed=seed)
            ax.plot(grid, proposal_density(grid, trained['edges']), alpha=.6, lw=1,
                    label=f'seed {seed}')
        ax.set(xlabel='x', ylabel='Function / proposal density', title=title)
    axes[0].legend(fontsize=8, ncol=2)
    plt.show()


def show_bin_scores(edges, scores, updated):
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), constrained_layout=True)
    centers = (edges[:-1]+edges[1:])/2
    axes[0].bar(centers, scores, width=np.diff(edges), color='#007c91', alpha=.8)
    axes[0].set(xlabel='x interval', ylabel='Mean squared weight', title='Scores from one training batch', xlim=(0, 1))
    axes[1].plot(edges, np.zeros_like(edges), '|', markersize=22, color='#526579')
    axes[1].plot(updated, np.ones_like(updated), '|', markersize=22, color='#007c91')
    axes[1].axvline(.63, color='#d16a20', ls=':', label='Peak center')
    axes[1].set(xlabel='Bin boundary', yticks=[0, 1], yticklabels=['Before', 'After'],
                ylim=(-.5, 1.5), xlim=(0, 1), title='Edges after one update')
    axes[1].legend(fontsize=10)
    plt.show()

def show_target_2d():
    with plt.rc_context({"font.size": 12}):
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.1), constrained_layout=True)
        x = np.linspace(XMIN, XMAX, 401)
        y = np.linspace(YMIN, YMAX, 401)
        xx, yy = np.meshgrid(x, y)
        im = axes[0].pcolormesh(x, y, f(xx, yy), shading="auto", cmap="GnBu")
        fig.colorbar(im, ax=axes[0], label=r"$f(x,y)$")
        axes[0].set(title="The target function", xlabel=r"$x$", ylabel=r"$y$",
                    xlim=(XMIN, XMAX), ylim=(YMIN, YMAX))
        axes[0].set_aspect("equal")

        y_slice = components()[0][2]
        xs = np.linspace(XMIN, XMAX, 1601)
        axes[1].plot(xs, f(xs, y_slice), color="#17365d", lw=2.2, label="Sum")
        for i, (a, mx, my, sx, sy) in enumerate(components()):
            values = a * np.exp(-0.5 * (((xs-mx)/sx)**2 + ((y_slice-my)/sy)**2))
            axes[1].plot(xs, values, color=("#007c91", "#d97732")[i % 2],
                         lw=1.5, ls="--", label=f"Component {i+1}")
        axes[1].set(title=fr"Slice at $y={y_slice:g}$", xlabel=r"$x$",
                    ylabel=r"$f(x,y)$", xlim=(XMIN, XMAX), ylim=(0, None))
        axes[1].legend(frameon=False, fontsize=10)
        axes[1].grid(alpha=0.15)
        plt.show()


def show_product_map(xedges, yedges):
    with plt.rc_context({"font.size": 12}):
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.1), constrained_layout=True)
        x = np.linspace(XMIN, XMAX, 401)
        y = np.linspace(YMIN, YMAX, 401)
        xx, yy = np.meshgrid(x, y)
        target = axes[0].pcolormesh(x, y, f(xx, yy), shading="auto", cmap="GnBu")
        fig.colorbar(target, ax=axes[0], label=r"$f(x,y)$")

        dx, dy = np.diff(xedges), np.diff(yedges)
        q = 1.0 / (len(dx) * len(dy) * np.outer(dx, dy))
        proposal = axes[1].pcolormesh(xedges, yedges, q.T, shading="flat", cmap="GnBu")
        fig.colorbar(proposal, ax=axes[1], label=r"$q(x,y)$")
        axes[1].vlines(xedges, YMIN, YMAX, color="#17365d", alpha=0.18, lw=0.4)
        axes[1].hlines(yedges, XMIN, XMAX, color="#17365d", alpha=0.18, lw=0.4)
        for ax, title in zip(axes, ("Target function", "Learned proposal density")):
            ax.set(title=title, xlabel=r"$x$", ylabel=r"$y$",
                   xlim=(XMIN, XMAX), ylim=(YMIN, YMAX))
            ax.set_aspect("equal")
        plt.show()


def show_rejection_batch(x, y, keep):
    x, y, keep = np.asarray(x), np.asarray(y), np.asarray(keep, dtype=bool)
    naccepted = int(keep.sum())
    with plt.rc_context({"font.size": 12}):
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.1), constrained_layout=True)
        axes[0].scatter(x[~keep], y[~keep], s=5, color="#9aa5af", alpha=0.35,
                        linewidths=0, label=f"Rejected: {len(x)-naccepted:,}")
        axes[0].scatter(x[keep], y[keep], s=7, color="#d97732", alpha=0.75,
                        linewidths=0, label=f"Accepted: {naccepted:,}")
        axes[0].legend(frameon=False, fontsize=10, loc="upper right", markerscale=2)
        axes[1].scatter(x[keep], y[keep], s=7, color="#d97732", alpha=0.75,
                        linewidths=0)
        titles = (f"Proposals: {len(x):,}", f"Accepted events: {naccepted:,}")
        for ax, title in zip(axes, titles):
            ax.set(title=title, xlabel=r"$x$", ylabel=r"$y$",
                   xlim=(XMIN, XMAX), ylim=(YMIN, YMAX))
            ax.set_aspect("equal")
        plt.show()


def show_cost(bf, vg):
    results = (bf, vg)
    training = np.array([result["nTrainEval"] for result in results])
    production = np.array([result["nFunEval"] for result in results])
    totals = np.array([result["nTotalEval"] for result in results])
    with plt.rc_context({"font.size": 12}):
        fig, ax = plt.subplots(figsize=(10, 3.1), constrained_layout=True)
        ax.barh([0, 1], training, height=0.5, color="#d97732", label="Training")
        ax.barh([0, 1], production, left=training, height=0.5,
                color="#007c91", label="Production")
        ax.set_yticks([0, 1], labels=["Uniform", "Adaptive"])
        ax.invert_yaxis()
        ax.set(xlabel=r"Evaluations of $f$", title="Total cost of producing the events")
        ax.set_xlim(0, 1.03 * max(totals))
        ax.set_xticks(np.linspace(0, max(totals), 4))
        ax.xaxis.set_major_formatter("{x:,.0f}")
        ax.grid(axis="x", alpha=0.15)
        ax.set_axisbelow(True)
        for i, result in enumerate(results):
            detail = (f"{result['nTotalEval']:,} total evaluations\n"
                      f"{len(result['events']):,} returned events\n"
                      f"{result['nAccepted']:,} accepted in full batches")
            ax.text(1.025, i, detail, transform=ax.get_yaxis_transform(),
                    va="center", ha="left", fontsize=10, color="#17365d")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.24),
                  ncol=2, frameon=False, fontsize=11)
        plt.show()


def show_offset_target():
    x = np.linspace(-5.0, 5.0, 401)
    y = np.linspace(-5.0, 5.0, 401)
    xx, yy = np.meshgrid(x, y)
    values = (np.exp(-0.5 * ((xx-2.5)**2 + (yy+2.5)**2))
              + np.exp(-0.5 * ((xx+2.5)**2 + (yy-2.5)**2)))
    with plt.rc_context({"font.size": 12}):
        fig, ax = plt.subplots(figsize=(10, 3.1), constrained_layout=True)
        im = ax.pcolormesh(x, y, values, shading="auto", cmap="GnBu")
        fig.colorbar(im, ax=ax, label=r"$f(x,y)$")
        ax.set(title="Two offset peaks: where should proposals go?", xlabel=r"$x$",
               ylabel=r"$y$", xlim=(-5, 5), ylim=(-5, 5))
        ax.set_aspect("equal")
        plt.show()


In [ ]:
def toy_integral(sigma):
    return 0.1 + 0.45*(erf((1-0.63)/(np.sqrt(2)*sigma))
                      - erf(-0.63/(np.sqrt(2)*sigma)))


In [ ]:
def summarize_weights(weights):
    weights = np.asarray(weights)
    if weights.ndim != 1 or len(weights) < 2 or not np.all(np.isfinite(weights)):
        raise ValueError('Use at least two finite, independent weights.')
    return float(weights.mean()), float(weights.std(ddof=1)/np.sqrt(len(weights)))


## A peak with a known area

On $0\leq x\leq 1$, use a Gaussian peak on a constant background:
$$g_\sigma(x)=0.1+\frac{0.9}{\sigma\sqrt{2\pi}}
\exp\left[-\frac{(x-0.63)^2}{2\sigma^2}\right].$$

The peak contains almost all of its Gaussian area inside this interval when $\sigma=0.12$. The exact finite-interval area is about $0.9991$.


In [ ]:
def toy(x, sigma=0.12):
    """A positive background plus a Gaussian peak on [0, 1]."""
    return 0.1 + 0.9*np.exp(-0.5*((x-0.63)/sigma)**2)/(sigma*np.sqrt(2*np.pi))


## Uniform Monte Carlo integration

Sample positions uniformly and average the function values. Multiply by the interval length:
$$\widehat I=(b-a)\overline g,\qquad
\operatorname{SE}(\widehat I)=(b-a)\frac{s_g}{\sqrt N}.$$


In [ ]:
rng = np.random.default_rng(142)
x_uniform = rng.uniform(0, 1, size=100)
values = toy(x_uniform, sigma=0.12)
estimate, error = values.mean(), values.std(ddof=1)/np.sqrt(len(values))
show_toy_samples(x_uniform, sigma=0.12)


## Make the peak much narrower

Keep the same 100 sample positions and reduce the width to 0.003. The exact area is now 1.

Look at the full interval and then the zoom. How many sampled heights can tell us about the peak?


In [ ]:
values = toy(x_uniform, sigma=0.003)
print(f"Area = {values.mean():.3f} ± {values.std(ddof=1)/np.sqrt(len(values)):.3f}")
show_toy_samples(x_uniform, sigma=0.003)


## Repeat the experiment

Each point below uses 100 new uniform samples. Bars show one estimated standard error.

A run that barely samples the peak can underestimate both its area and its uncertainty. Increasing the sample count helps, but most evaluations still land away from the peak.


In [ ]:
broad_runs = repeated_uniform(0.12)
narrow_runs = repeated_uniform(0.003)
show_uniform_repeats(broad_runs, narrow_runs)


## Choose where to sample

Put more sample positions near the peak. Call their probability density $q(x)$.

We will first place the bin edges by hand to see how this works. Each interval has the same probability, so narrower intervals have higher density:
$$q(x)=\frac{1}{K\,\Delta x_i}\quad\text{inside interval }i.$$

We still need positive sampling density throughout the integration domain.


In [ ]:
def draw_1d(edges, size, rng):
    widths = np.diff(edges)
    if len(widths) < 1 or np.any(widths <= 0):
        raise ValueError('Bin edges must increase strictly.')
    index = rng.integers(len(widths), size=size)
    x = edges[index] + widths[index]*rng.random(size)
    q = 1/(len(widths)*widths[index])
    return x, q, index


In [ ]:
manual_edges = np.array([0, .4, .60, .62, .626, .634, .64, .66, .8, 1.])
x_manual, q_manual, _ = draw_1d(manual_edges, 1000, np.random.default_rng(143))
show_bins(manual_edges, x_manual)


## Correct for the sampling density

The ordinary mean now estimates $\int q(x)g(x)\,dx$. Divide each value by the density that produced its sample:
$$I=\int q(x)\frac{g(x)}{q(x)}\,dx,
\qquad\widehat I=\frac{1}{N}\sum_{j=1}^{N}\frac{g(x_j)}{q(x_j)}.$$

This is **importance sampling**. Its uncertainty is the sample standard deviation of the weights divided by $\sqrt N$.


In [ ]:
weights_manual = toy(x_manual, .003)/q_manual
estimate, error = summarize_weights(weights_manual)
print(f"Uncorrected mean: {toy(x_manual, .003).mean():.3f}")
print(f"Weighted estimate: {estimate:.3f} ± {error:.3f}; exact area = 1")


## Look at what enters the average

Both panels use 1,000 function evaluations. A few large weights make the uniform estimate fluctuate.

The unequal bins distribute the important contribution over more samples. We now need a way to learn those bins from evaluations of the function.


In [ ]:
x_check = np.random.default_rng(144).random(1000)
show_weight_spread(toy(x_check, .003), weights_manual)


## Sampling an adaptive grid

Choose an interval uniformly, then a position uniformly within that interval. Return the density along with the point so that every weight uses the correct $q$.

The bin widths change during training; the probability of choosing each bin remains $1/K$.


In [ ]:
def draw_1d(edges, size, rng):
    widths = np.diff(edges)
    if len(widths) < 1 or np.any(widths <= 0):
        raise ValueError('Bin edges must increase strictly.')
    index = rng.integers(len(widths), size=size)
    x = edges[index] + widths[index]*rng.random(size)
    q = 1/(len(widths)*widths[index])
    return x, q, index


## VEGAS learns the bin boundaries

VEGAS repeatedly samples, scores the intervals, and moves their edges. Intervals with large squared weights receive more grid resolution.

We implement its adaptive importance-sampling component here. Production samples will be independent draws from the final grid.


In [ ]:
def bin_scores(index, weights, bins):
    counts = np.bincount(index, minlength=bins)
    if np.any(counts == 0):
        raise ValueError('An adaptation bin is empty; increase the training batch.')
    return np.bincount(index, weights=weights**2, minlength=bins)/counts


In [ ]:
def smooth_and_compress(scores, alpha_damp):
    """Lepage (2021), Eqs. 18-19: neighbor smoothing and compression."""
    if np.any(scores <= 0) or not np.all(np.isfinite(scores)):
        raise ValueError('Adaptation requires finite positive bin scores.')
    if alpha_damp is not None and alpha_damp < 0:
        raise ValueError('alpha_damp must be nonnegative or None.')
    smoothed = np.convolve(np.pad(scores, (1, 1), mode='edge'),
                           np.array([1, 6, 1])/8, mode='valid')
    z = smoothed/smoothed.sum()
    if alpha_damp is not None:
        z = ((z-1)/np.log(z))**alpha_damp
    return z


In [ ]:
def rebin(edges, scores, alpha_damp):
    """Allocate equal cumulative score to each new interval."""
    z = smooth_and_compress(scores, alpha_damp)
    cumulative = np.r_[0., np.cumsum(z)]
    new = np.interp(np.linspace(0, cumulative[-1], len(edges)), cumulative, edges)
    if np.any(np.diff(new) <= 0):
        raise ValueError('Adaptation produced a collapsed interval.')
    return new


In [ ]:
def train_1d(target, bins=20, batch=1000, iterations=6, alpha_damp=1.5, seed=None):
    if bins < 2 or batch < bins or iterations < 0:
        raise ValueError('Require bins >= 2, batch >= bins, and iterations >= 0.')
    edges = np.linspace(0, 1, bins+1)
    history = [edges.copy()]
    rng = np.random.default_rng(seed)
    for _ in range(iterations):
        x, q, index = draw_1d(edges, batch, rng)
        weights = target(x)/q
        scores = bin_scores(index, weights, bins)
        edges = rebin(edges, scores, alpha_damp)
        history.append(edges.copy())
    return {'edges': edges, 'history': history, 'nTrainEval': batch*iterations}


## Squared weights identify costly intervals

Estimate $d_i=\operatorname{mean}(w^2\mid i)$ in each interval, where $w=g/q$.

Large scores identify intervals that contribute strongly to the uncertainty. Redistributing the edges puts more intervals there. The update smooths and compresses the scores; we will examine that stabilization after seeing the method work.


In [ ]:
initial_edges = np.linspace(0, 1, 21)
x0, q0, index0 = draw_1d(initial_edges, 1000, np.random.default_rng(145))
scores0 = bin_scores(index0, toy(x0, .003)/q0, 20)
updated_edges = rebin(initial_edges, scores0, alpha_damp=1.5)
show_bin_scores(initial_edges, scores0, updated_edges)


## Watch the learned grid change

These are grids produced by the training loop. The orange ticks mark bin boundaries; the teal curve is the sampling density.

Watch the bins collect around the narrow peak. Training can only react to features that its samples have encountered.


In [ ]:
training_seed, production_seed = np.random.SeedSequence(142).spawn(2)
learned = train_1d(lambda x: toy(x, .003), seed=training_seed)
show_grid_history(learned['history'])


## Freeze the grid and estimate the integral

Use fresh production samples after training. Every production point contributes one weight, and the grid stays fixed while we estimate the mean and its uncertainty.


In [ ]:
def estimate_1d(target, edges, size, seed=None):
    x, q, index = draw_1d(edges, size, np.random.default_rng(seed))
    weights = target(x)/q
    estimate, error = summarize_weights(weights)
    return {'estimate': estimate, 'error': error, 'nFunEval': size,
            'x': x, 'q': q, 'weights': weights}


In [ ]:
production = estimate_1d(lambda x: toy(x, .003), learned['edges'], 2000, production_seed)
print(f"Integral = {production['estimate']:.4f} ± {production['error']:.4f}")
print(f"Evaluations: {learned['nTrainEval']} training + {production['nFunEval']} production")


## Compare at the same total cost

Each adaptive run spends 6,000 evaluations learning a grid and 2,000 evaluating the final estimate. Each uniform run gets all 8,000 evaluations.

Every run trains independently. Compare the spread across runs and the reported uncertainty; the cost of training is included.


In [ ]:
uniform_cost, adaptive_cost = compare_equal_cost()
show_equal_cost(uniform_cost, adaptive_cost)


## The original two-Gaussian problem

Now use the lab's function on $0<x,y<5$:
$$f(x,y)=\sum_{k=1}^{2}A_k\exp\left[-\frac12
\left(\frac{(x-\mu_{kx})^2}{\sigma_{kx}^2}
+\frac{(y-\mu_{ky})^2}{\sigma_{ky}^2}\right)\right].$$

The components have the same center. One is broad; the other is tall and narrow.


In [ ]:
A1, A2 = 1.0, 4.0
MU1X = MU1Y = MU2X = MU2Y = 2.5
SIGMA1X = SIGMA1Y = 1.0
SIGMA2X = SIGMA2Y = 0.1
XMIN = YMIN = 0.0
XMAX = YMAX = 5.0


In [ ]:
def components():
    return ((A1, MU1X, MU1Y, SIGMA1X, SIGMA1Y),
            (A2, MU2X, MU2Y, SIGMA2X, SIGMA2Y))


In [ ]:
def f(x, y):
    return sum(a * np.exp(-0.5 * (((x-mx)/sx)**2 + ((y-my)/sy)**2))
               for a, mx, my, sx, sy in components())


In [ ]:
def exact_integral():
    """Independent finite-domain Gaussian reference using error functions."""
    total = 0.
    for a, mx, my, sx, sy in components():
        ix = sx*np.sqrt(np.pi/2)*(erf((XMAX-mx)/(np.sqrt(2)*sx))
                                - erf((XMIN-mx)/(np.sqrt(2)*sx)))
        iy = sy*np.sqrt(np.pi/2)*(erf((YMAX-my)/(np.sqrt(2)*sy))
                                - erf((YMIN-my)/(np.sqrt(2)*sy)))
        total += a*ix*iy
    return total


## Inspect the target before sampling

The heatmap shows the function itself. The slice through its center resolves the narrow peak.

Peak height and area carry different information: the tall narrow component contributes about 4% of this function's total integral.


In [ ]:
def f(x, y):
    return sum(a * np.exp(-0.5 * (((x-mx)/sx)**2 + ((y-my)/sy)**2))
               for a, mx, my, sx, sy in components())


In [ ]:
show_target_2d()


## A sampling density in two dimensions

VEGAS learns a separate grid for each coordinate:
$$q(x,y)=q_x(x)q_y(y)=\frac{1}{K_xK_y\,\Delta x_i\Delta y_j}.$$

The density must use the width of the selected interval on each axis.


In [ ]:
def draw_proposals(xLow, yLow, size, rng):
    """Choose each interval equally often, then sample within its own width."""
    dx, dy = np.diff(xLow), np.diff(yLow)
    ix = rng.integers(len(dx), size=size)
    iy = rng.integers(len(dy), size=size)
    x = xLow[ix] + dx[ix] * rng.random(size)
    y = yLow[iy] + dy[iy] * rng.random(size)
    q = 1.0 / (len(dx) * len(dy) * dx[ix] * dy[iy])
    return x, y, q


## Update both coordinate grids

The same squared weights score the x intervals and the y intervals. A point contributes to one interval on each axis.

After each batch, update both grids and repeat.


In [ ]:
def setup_intervals(NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    """Adapt the independent coordinate maps used by VEGAS."""
    if NN < 2 or KK < NN or nIterations < 0:
        raise ValueError('Require NN >= 2, KK >= NN, and nIterations >= 0.')
    if not (XMIN < XMAX and YMIN < YMAX):
        raise ValueError('Each domain must have positive width.')
    xLow = np.linspace(XMIN, XMAX, NN+1)
    yLow = np.linspace(YMIN, YMAX, NN+1)
    rng = np.random.default_rng(seed)
    for _ in range(nIterations):
        x, y, q = draw_proposals(xLow, yLow, KK, rng)
        weights = f(x, y)/q
        ix = np.searchsorted(xLow, x, side='right') - 1
        iy = np.searchsorted(yLow, y, side='right') - 1
        xLow = rebin(xLow, bin_scores(ix, weights, NN), alpha_damp)
        yLow = rebin(yLow, bin_scores(iy, weights, NN), alpha_damp)
    return xLow, yLow, np.diff(xLow), np.diff(yLow)


## The learned product density

Compare the target with the learned proposal. The grid can concentrate around the common center while still covering the whole domain.

The two functions do not have to be identical: the weights account for the remaining difference.


In [ ]:
grid_seed, integral_seed = np.random.SeedSequence(242).spawn(2)
x_edges, y_edges, _, _ = setup_intervals(seed=grid_seed)
show_product_map(x_edges, y_edges)


## Estimate the two-dimensional integral

With uniform sampling, $q=1/25$ and each weight is $25f$. With the learned grid, each weight is $f/q$.

Below, both production estimates use 5,000 fresh samples. The adaptive method also used 40,000 training evaluations; this plot compares production weights, not equal total cost.


In [ ]:
def estimate_integral(xLow, yLow, size, seed=None):
    x, y, q = draw_proposals(xLow, yLow, size, np.random.default_rng(seed))
    weights = f(x, y)/q
    estimate, error = summarize_weights(weights)
    return {'estimate': estimate, 'error': error, 'nFunEval': size, 'weights': weights}


In [ ]:
uniform_2d = estimate_integral(np.array([0., 5.]), np.array([0., 5.]), 5000, seed=243)
adaptive_2d = estimate_integral(x_edges, y_edges, 5000, seed=integral_seed)
show_integral_2d(uniform_2d, adaptive_2d, exact_integral())


## Generate events distributed like the target

An event sample should follow the normalized density $p(x,y)=f(x,y)/I$.

Proposals follow $q$. Accept each proposal with probability
$$P(\mathrm{accept}\mid x,y)=\frac{f(x,y)}{M q(x,y)},
\qquad M\geq\max_{x,y}\frac{f(x,y)}{q(x,y)}.$$

Then $q\,P(\mathrm{accept})=f/M$: accepted events have the desired shape, and every retained event has equal weight.


## Try uniform rejection first

Draw a batch uniformly over the square. Keep a proposal when a new random number lies below its acceptance probability.

The accepted points should follow the broad shape and the narrow peak.


In [ ]:
def rejection_bound(xLow, yLow):
    """Conservative global bound on f/q for this positive Gaussian mixture.

    Each component is largest at the point in a cell closest to its mean.
    The sum of those component maxima bounds the mixture throughout that cell.
    """
    upper = np.zeros((len(xLow)-1, len(yLow)-1))
    for a, mx, my, sx, sy in components():
        if a < 0 or sx <= 0 or sy <= 0:
            raise ValueError('Require nonnegative amplitudes and positive widths.')
        cx = np.clip(mx, xLow[:-1], xLow[1:])
        cy = np.clip(my, yLow[:-1], yLow[1:])
        upper += a*np.exp(-0.5*(((cx[:,None]-mx)/sx)**2 + ((cy[None,:]-my)/sy)**2))
    cell_inverse_q = (len(xLow)-1)*(len(yLow)-1)*np.outer(np.diff(xLow), np.diff(yLow))
    bound = float(np.max(upper * cell_inverse_q))
    if not np.isfinite(bound) or bound <= 0:
        raise ValueError('The rejection bound must be finite and positive.')
    return np.nextafter(bound, np.inf)


In [ ]:
rng_events = np.random.default_rng(142)
box_x, box_y = np.array([0., 5.]), np.array([0., 5.])
x_batch, y_batch, q_batch = draw_proposals(box_x, box_y, 4096, rng_events)
bound = rejection_bound(box_x, box_y)
keep = rng_events.random(len(x_batch)) < f(x_batch, y_batch)/(bound*q_batch)
show_rejection_batch(x_batch, y_batch, keep)


## The rejection bound must cover every weight

An observed maximum only covers the points already sampled. For this Gaussian sum, we can bound each component throughout every grid cell and add those bounds.

The code uses that conservative envelope. If a weight exceeds it, sampling stops with an error; clipping the acceptance probability would change the distribution.

Finer cells can also tighten this bound, so compare the envelope along with the proposal shape.


In [ ]:
for label, xe, ye in [('Uniform', box_x, box_y), ('Learned grid', x_edges, y_edges)]:
    x_test, y_test, q_test = draw_proposals(xe, ye, 5000, np.random.default_rng(144))
    print(f"{label}: observed maximum = {(f(x_test, y_test)/q_test).max():.3f}; "
          f"conservative bound = {rejection_bound(xe, ye):.3f}")


In [ ]:
def sample_events(nPoints, xLow, yLow, seed=None):
    """Independent rejection sampling with a proved Gaussian-mixture envelope."""
    if nPoints < 1:
        raise ValueError('nPoints must be positive.')
    rng = np.random.default_rng(seed)
    bound = rejection_bound(xLow, yLow)
    accepted, proposals = [], []
    naccepted = neval = 0
    maximum = 0.0
    while naccepted < nPoints:
        x, y, q = draw_proposals(xLow, yLow, 4096, rng)
        weights = f(x, y)/q
        maximum = max(maximum, float(weights.max()))
        if np.any(weights > bound*(1+1e-12)):
            raise ValueError('Envelope violated; these samples are invalid.')
        keep = rng.random(len(x)) < weights/bound
        accepted.append(np.column_stack((x[keep], y[keep])))
        proposals.append(np.column_stack((x, y)))
        naccepted += int(keep.sum())
        neval += len(x)
    return {'events': np.concatenate(accepted)[:nPoints],
            'proposals': np.concatenate(proposals), 'nFunEval': neval,
            'nAccepted': naccepted, 'maxWeightEncountered': maximum, 'bound': bound}


In [ ]:
def brute_force(nPoints, seed=None):
    result = sample_events(nPoints, np.array([XMIN, XMAX]), np.array([YMIN, YMAX]), seed)
    return {**result, 'yy1': result['events'][:,0], 'yy2': result['events'][:,1],
            'nTrainEval': 0, 'nTotalEval': result['nFunEval']}


In [ ]:
def vegas(nPoints, NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    train_seed, sample_seed = np.random.SeedSequence(seed).spawn(2)
    xLow, yLow, dx, dy = setup_intervals(NN, KK, nIterations, alpha_damp, train_seed)
    result = sample_events(nPoints, xLow, yLow, sample_seed)
    return {**result, 'yy1vegas': result['events'][:,0], 'yy2vegas': result['events'][:,1],
            'yy1vrho': result['proposals'][:,0], 'yy2vrho': result['proposals'][:,1],
            'vegasRatioMax': result['bound'], 'xLow': xLow, 'yLow': yLow,
            'nTrainEval': KK*nIterations, 'nTotalEval': KK*nIterations+result['nFunEval']}


In [ ]:
def lego_plot(xAmplitudes, yAmplitudes, nBins, xLabel, yLabel, title):
    hist, xe, ye = np.histogram2d(xAmplitudes, yAmplitudes, bins=nBins,
                                range=((XMIN,XMAX),(YMIN,YMAX)))
    xx, yy = np.meshgrid(xe[:-1], ye[:-1], indexing='ij')
    fig = plt.figure(figsize=(7,4.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.bar3d(xx.ravel(), yy.ravel(), np.zeros(hist.size),
             np.diff(xe)[0], np.diff(ye)[0], hist.ravel(), color='#007c91')
    ax.set(xlabel=xLabel, ylabel=yLabel, zlabel='Count', title=title,
           xlim=(XMIN,XMAX), ylim=(YMIN,YMAX))
    plt.show()
    return ax


In [ ]:
def plot_results(nPoints=100_000, nBins=40, NN=40, KK=2000,
                 nIterations=20, alpha_damp=1.5, seed=142):
    bf = brute_force(nPoints, seed)
    vg = vegas(nPoints, NN, KK, nIterations, alpha_damp, seed)
    lego_plot(bf['yy1'], bf['yy2'], nBins, 'x', 'y', 'Uniform rejection: accepted events')
    lego_plot(vg['yy1vrho'], vg['yy2vrho'], nBins, 'x', 'y', 'Adaptive proposals q(x,y)')
    lego_plot(vg['yy1vegas'], vg['yy2vegas'], nBins, 'x', 'y', 'Adaptive rejection: accepted events')
    for name, result in [('Uniform',bf), ('Adaptive',vg)]:
        print(f"{name}: training={result['nTrainEval']:,}; production={result['nFunEval']:,}; "
              f"total={result['nTotalEval']:,}; returned events={nPoints:,}")
        print(f"  maximum weight={result['maxWeightEncountered']:.6g}; "
              f"bound={result['bound']:.6g}")
    return bf, vg


## Uniform rejection: 100,000 events

Repeat the rejection step until 100,000 events have been collected. The histogram shows event counts in each two-dimensional bin.

We will compare the accepted sample with the adaptive proposal and its accepted sample.


In [ ]:
bf = brute_force(100_000, seed=142)
vg = vegas(100_000, seed=142)
lego_plot(bf['yy1'], bf['yy2'], 40, 'x', 'y', 'Uniform rejection: accepted events');


## Adaptive proposals

The trained grid puts more proposed points near the center. These points are distributed according to the learned density q.

The next rejection step corrects the remaining difference between the proposal and the target.


In [ ]:
lego_plot(vg['yy1vrho'], vg['yy2vrho'], 40, 'x', 'y', 'Adaptive proposals');


## Adaptive rejection: 100,000 events

After rejection, the accepted histogram should reproduce the same shape as the uniform-rejection result.

Changing the proposal changes how much work is needed to obtain the events.


In [ ]:
lego_plot(vg['yy1vegas'], vg['yy2vegas'], 40, 'x', 'y', 'Adaptive rejection: accepted events');


## Count all function evaluations

Include training and production, including the final batch's surplus events. Wall-clock time also depends on the integrand and sampling code.


In [ ]:
show_cost(bf, vg)
for name, result in [('Uniform', bf), ('Adaptive', vg)]:
    print(f"{name}: maximum weight = {result['maxWeightEncountered']:.3f}; bound = {result['bound']:.3f}")


## Noisy scores can move the grid too far

Return to the narrow one-dimensional peak. Each curve uses a different seed and six training batches of 500 points.

Compare the final grids with and without score compression. Removing compression makes each batch's large scores more influential; it can make the next grid less useful.


In [ ]:
show_damping()


## Control how strongly a batch moves the grid

After smoothing, normalize the scores to $z_i$ and use $[(z_i-1)/\log z_i]^\alpha$ to allocate the new intervals.

Smaller positive $\alpha$ slows adaptation. $\alpha=0$ leaves the grid fixed. `None` removes this compression while retaining neighbor smoothing.


In [ ]:
def smooth_and_compress(scores, alpha_damp):
    """Lepage (2021), Eqs. 18-19: neighbor smoothing and compression."""
    if np.any(scores <= 0) or not np.all(np.isfinite(scores)):
        raise ValueError('Adaptation requires finite positive bin scores.')
    if alpha_damp is not None and alpha_damp < 0:
        raise ValueError('alpha_damp must be nonnegative or None.')
    smoothed = np.convolve(np.pad(scores, (1, 1), mode='edge'),
                           np.array([1, 6, 1])/8, mode='valid')
    z = smoothed/smoothed.sum()
    if alpha_damp is not None:
        z = ((z-1)/np.log(z))**alpha_damp
    return z


## Exercise 1: remove the damping

Use 10,000 returned events for both runs. Compare `alpha_damp=1.5` with `alpha_damp=None`. Here `None` removes score compression; neighbor smoothing remains enabled.

```python
bf, vg = plot_results(10_000, alpha_damp=None, seed=142)
```

Compare the learned proposal, the rejection bound, and total function evaluations. Repeat with several seeds. Does a change that helps one run also help the others?

Use `alpha_damp=0` to leave the initial grid fixed.


## Separate the two peaks

Move the Gaussian centers to two separated positions inside the square. Both components now have amplitude 1 and width 1 on each axis.

Before running the sampler, predict where two independent coordinate grids will put proposals.


In [ ]:
show_offset_target()


## Exercise 2: offset the Gaussians

Set the parameters below, then rerun `plot_results`. The rejection bound is recomputed from the new function.

```python
A1 = A2 = 1.0
MU1X, MU1Y = 2.5, -2.5
MU2X, MU2Y = -2.5, 2.5
SIGMA1X = SIGMA1Y = SIGMA2X = SIGMA2Y = 1.0
XMIN = YMIN = -5.0
XMAX = YMAX = 5.0
bf, vg = plot_results(100_000, seed=142)
```

Compare the target peaks with the proposal peaks. Which combinations of x and y receive proposals? Explain how the product $q_xq_y$ affects the cost of producing accepted events.


## What to inspect in each calculation

**Integration:** compare the estimate and its uncertainty with the known integral. Inspect the weight spread and repeat with independent seeds.

**Event generation:** compare the accepted distribution with the target, verify the envelope, and count all evaluations.

**Adaptation:** inspect the learned bins and proposal. A product of one-dimensional densities can concentrate along each axis while wasting samples on combinations where the target is small.


## References

1. T. Barklow, "Monte Carlo Techniques", https://courses.physics.illinois.edu/phys524/fa2023/topics.html (2023).
2. G. P. Lepage, "A New Algorithm for Adaptive Multidimensional Integration", J. Comput. Phys. 27, 192 (1978).
3. G. P. Lepage, ["Adaptive multidimensional integration: VEGAS enhanced"](https://arxiv.org/abs/2009.05112), J. Comput. Phys. 439, 110386 (2021).

The grid update follows Section 2 of the third reference. This notebook isolates adaptive importance sampling with independent production draws; full VEGAS implementations also use stratification, and VEGAS+ adds adaptive stratification.
